In [0]:
%sql
CREATE TABLE IF NOT EXISTS migration.silver.customers (
    customer_id BIGINT,
    customer_name STRING,
    email STRING,
    city STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _source_batch_id STRING,
    _source_run_id STRING,
    _source_ingestion_timestamp TIMESTAMP,
    _processed_timestamp TIMESTAMP
)
USING DELTA;

## Find the latest version of every customer

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW latest_customers AS
WITH ranked_customers AS (

    SELECT
        customer_id,
        customer_name,
        email,
        city,
        created_at,
        updated_at,
        _batch_id,
        _run_id,
        _ingestion_timestamp,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, _ingestion_timestamp DESC
        ) AS rn

    FROM migration.bronze.customers
)

SELECT
    customer_id,
    customer_name,
    email,
    city,
    created_at,
    updated_at,
    _batch_id,
    _run_id,
    _ingestion_timestamp
FROM ranked_customers
WHERE rn = 1;

-- SELECT * from ranked_customers;

# Validating/Checking Purpose

In [0]:
%sql
WITH latest AS (

    SELECT
        customer_id,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY updated_at DESC, _ingestion_timestamp DESC
        ) AS rn

    FROM migration.bronze.customers
)

SELECT
    customer_id,
    COUNT(*) AS cnt
FROM latest
WHERE rn = 1
GROUP BY customer_id
HAVING COUNT(*) > 1;

# MERGE into Silver

In [0]:
%sql
MERGE INTO migration.silver.customers AS target

USING latest_customers AS source

ON target.customer_id = source.customer_id

WHEN MATCHED
AND source.updated_at > target.updated_at
THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.email = source.email,
    target.city = source.city,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target._source_batch_id = source._batch_id,
    target._source_run_id = source._run_id,
    target._source_ingestion_timestamp = source._ingestion_timestamp,
    target._processed_timestamp = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    customer_id,
    customer_name,
    email,
    city,
    created_at,
    updated_at,
    _source_batch_id,
    _source_run_id,
    _source_ingestion_timestamp,
    _processed_timestamp
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.email,
    source.city,
    source.created_at,
    source.updated_at,
    source._batch_id,
    source._run_id,
    source._ingestion_timestamp,
    current_timestamp()
);

# Verify

In [0]:
%sql
SELECT *
FROM migration.silver.customers
ORDER BY customer_id;

In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS cnt
FROM migration.silver.customers
GROUP BY customer_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM migration.silver.customers
WHERE customer_id = 1;